In [0]:
#Leitura do gabarito vindo da bronze
df_leitura = spark.table("gabarito")
display(df_leitura)

In [0]:
from pyspark.sql import functions as F

df_bronze_gab = spark.table("gabarito")

#caracteres para remover acentos 
acentos_origem  = "ÁÀÃÂÄÉÈÊËÍÌÎÏÓÒÕÔÖÚÙÛÜÇÑ"
acentos_destino = "AAAAAEEEEIIIIOOOOOUUUUCN"

#criacao da tabela a partir da tabea anterioe
df_silver_gabarito = (
    df_bronze_gab
    #padronização da chave da guia
    .withColumn(
        "id_guia", 
        F.lower(F.trim(F.col("id_guia")))
    )
    
    # 2. padronização dos nomes sem espaços extras e sem acentos (ex: Chloé -> CHLOE)
    .withColumn(
        "nome",
        F.translate(F.upper(F.trim(F.col("nome"))), acentos_origem, acentos_destino)
    )
    
    #carteirinha apenas com números (mantendo zeros à esquerda)
    .withColumn(
        "carteirinha",
        F.regexp_replace(F.trim(F.col("carteirinha")), r"[^0-9]", "")
    )
    
    #colocando o codigo TUSS limpo e normalizado para o padrão ANS de 8 dígitos (remove '00' inicial se tiver 10 dígitos)
    .withColumn("tuss_limpo", F.regexp_replace(F.trim(F.col("codigo_tuss")), r"[^0-9]", ""))
    .withColumn(
        "codigo_tuss",
        #remove '00' inicial se tiver 10 dígitos
        F.regexp_replace(F.col("tuss_limpo"), r"^00(\d{8})$", "$1")
    )
    
    #padronizar em letras maiusculas, removendo primeiramente as aspas duplas e depois retira as duplas e no final retira os acentos
    .withColumn(
        "descricao",
        F.translate(F.upper(F.trim(F.regexp_replace(F.regexp_replace(F.col("descricao"), '"', ''), r"\s+", " "))),acentos_origem, acentos_destino)
    )
    .select(
        "id_guia",
        "item",
        "nome",
        "carteirinha",
        "codigo_tuss",
        "descricao"
    )
)

df_silver_gabarito.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_gabarito")
display(df_silver_gabarito)